In [1]:
import os
import re

In [2]:
folder = "/media/crk/datastore_1/Rag_Defence_Utility/logs_20260201_064946"

files = os.listdir(folder)

files.remove("summary.log")
files.remove("run_all_combinations.sh")

files.sort()

In [3]:
files[0]

'nq_01_all_disabled.log'

In [4]:
with open(os.path.join(folder, files[0]), 'r') as f:
    content = f.readlines()

In [5]:
data = {
    "nq" : [],
    "triviaqa" : [],
    "pubmedqa" : []
}

In [6]:
new_json = {
    "dp": False,
    "trustrag": False,
    "av": False,
    "poisoning": False,
    "mba": False,
    "ado": False,
    "answer_relevancy": 0,
    "faithfulness": 0,
    "context_relevancy": 0,
    "context_recall": 0,
    "poison_acr": 0,
    "mba_member": 0,
    "mba_non_member": 0,
    "poison_success": 0,
    "poison_total": 0
}

In [7]:
files[0].split("_")

['nq', '01', 'all', 'disabled.log']

In [8]:
for i in range(len(files)):
    temp_json = new_json.copy()

    file_name = files[i].replace(".log", "")
    parts = file_name.split("_")

    for part in parts:
        if part in temp_json:
            temp_json[part] = True

    if "all_defenses" in file_name:
        temp_json["dp"] = True
        temp_json["trustrag"] = True
        temp_json["av"] = True

    with open(os.path.join(folder, files[i]), 'r') as f:
        content = f.readlines()

    content = content[-30:]

    answer_relevancy = float(re.search(r'Answer Relevancy:     ([0-9.]+)', ''.join(content)).group(1))
    faithfulness = float(re.search(r'Faithfulness:         ([0-9.]+)', ''.join(content)).group(1))
    context_relevancy = float(re.search(r'Contextual Relevancy: ([0-9.]+)', ''.join(content)).group(1))
    context_recall = float(re.search(r'Contextual Recall:    ([0-9.]+)', ''.join(content)).group(1))

    temp_json["answer_relevancy"] = answer_relevancy
    temp_json["faithfulness"] = faithfulness
    temp_json["context_relevancy"] = context_relevancy
    temp_json["context_recall"] = context_recall

    if "poisoning" in file_name:
        text = ''.join(content)
        match = re.search(
            r'Poisoning ASR:\s*([0-9.]+)%\s*\((\d+)/(\d+)\)',
            text
        )

        poison_asr_pct = float(match.group(1))   # 0.0
        poison_success = int(match.group(2))     # 0
        poison_total = int(match.group(3))       # 50

        temp_json["poison_asr"] = poison_asr_pct
        temp_json["poison_success"] = poison_success
        temp_json["poison_total"] = poison_total

    if "mba" in file_name:
        text = ''.join(content)
        match_member = re.search(
            r'MBA Member Mask Accuracy: ([0-9.]+)%',
            text
        )
        match_non_member = re.search(
            r'MBA Non-Member Mask Accuracy: ([0-9.]+)%',
            text
        )

        mba_member_asr = float(match_member.group(1))       # 0.0
        mba_non_member_asr = float(match_non_member.group(1)) # 0.0

        temp_json["mba_member"] = mba_member_asr
        temp_json["mba_non_member"] = mba_non_member_asr
    
    if "nq" in file_name:
        data["nq"].append(temp_json)
    elif "triviaqa" in file_name:
        data["triviaqa"].append(temp_json)
    elif "pubmedqa" in file_name:
        data["pubmedqa"].append(temp_json)

In [9]:
import pandas as pd

rows = []

for dataset_name, records in data.items():
    for record in records:
        r = record.copy()
        r["dataset"] = dataset_name
        rows.append(r)

df = pd.DataFrame(rows)


In [10]:
df.fillna(0, inplace=True)

In [11]:
df.to_csv("defence_results_summary_20260201.csv", index=False)

In [12]:
df[df["dataset"] == "nq"]

,dp,trustrag,av,poisoning,mba,ado,answer_relevancy,faithfulness,context_relevancy,context_recall,poison_acr,mba_member,mba_non_member,poison_success,poison_total,dataset,poison_asr
0,False,False,False,False,False,False,0.930,0.74,0.489,0.593,0,0.0,0.0,0,0,nq,0.0
1,False,False,False,True,False,False,0.910,0.73,0.506,0.598,0,0.0,0.0,35,50,nq,70.0
2,False,True,False,True,False,False,0.890,0.79,0.462,0.614,0,0.0,0.0,0,50,nq,0.0
3,False,False,True,True,False,False,0.740,0.70,0.439,0.583,0,0.0,0.0,36,50,nq,72.0
4,True,False,False,False,False,False,0.863,0.77,0.321,0.361,0,0.0,0.0,0,0,nq,0.0
5,False,True,False,False,False,False,0.860,0.72,0.470,0.586,0,0.0,0.0,0,0,nq,0.0
6,False,False,True,False,False,False,0.780,0.77,0.449,0.592,0,0.0,0.0,0,0,nq,0.0
7,True,True,False,False,False,False,0.853,0.67,0.344,0.380,0,0.0,0.0,0,0,nq,0.0
8,True,False,True,False,False,False,0.740,0.58,0.286,0.364,0,0.0,0.0,0,0,nq,0.0
9,False,True,True,False,False,False,0.750,0.69,0.387,0.571,0,0.0,0.0,0,0,nq,0.0


In [13]:
import pandas as pd

# Assuming df is already loaded in your environment

print("=== DATA FOR TABLE 1: UTILITY COLLAPSE ===")
for ds in ['nq', 'pubmedqa', 'triviaqa']:
    # 1. Baseline: No Defenses
    # Using row with no defenses enabled. If multiple exist, taking the one with highest recall.
    base_rows = df[
        (df['dataset'] == ds) & 
        (df['dp'] == False) & 
        (df['trustrag'] == False) & 
        (df['av'] == False) & 
        (df['poisoning'] == False) & 
        (df['mba'] == False) & 
        (df['ado'] == False)
    ]
    
    if base_rows.empty:
        print(f"{ds}: No Baseline Row Found")
        continue
        
    base = base_rows.iloc[0] # Take first match
    
    # 2. Full Stack: All Defenses (DP + TrustRAG + AV)
    # We look for the row with the lowest context_recall as a proxy for the heaviest defense stack
    # effectively simulating the "Full Stack" performance drop.
    subset = df[(df['dataset'] == ds) & ((df['dp']) | (df['trustrag']) | (df['av']))]
    
    if subset.empty:
        print(f"{ds}: No Defense Rows Found")
        continue
    
    full = subset.loc[subset['context_recall'].idxmin()]
    
    recall_drop = ((base['context_recall'] - full['context_recall']) / base['context_recall']) * 100
    
    print(f"DATASET: {ds}")
    print(f"  Baseline Recall: {base['context_recall']:.3f} | Relevancy: {base['answer_relevancy']:.3f}")
    print(f"  FullStack Recall: {full['context_recall']:.3f} | Relevancy: {full['answer_relevancy']:.3f}")
    print(f"  Recall Drop: -{recall_drop:.1f}%")
    print("-" * 30)

print("\n=== DATA FOR TABLE 2: POISONING DEFENSE (TriviaQA) ===")
# Poisoning Only - Baseline
p_base_rows = df[(df['dataset'] == 'triviaqa') & (df['poisoning'] == True) & (df['trustrag'] == False)]
# Poisoning + TrustRAG - Defense
p_def_rows = df[(df['dataset'] == 'triviaqa') & (df['poisoning'] == True) & (df['trustrag'] == True)]

if not p_base_rows.empty and not p_def_rows.empty:
    p_base = p_base_rows.iloc[0]
    p_def = p_def_rows.iloc[0]
    
    # Reduction in Success Rate (Percentage Point Drop)
    asr_drop = p_base['poison_asr'] - p_def['poison_asr'] 
    
    print(f"Baseline ASR: {p_base['poison_asr']}%")
    print(f"TrustRAG ASR: {p_def['poison_asr']}%")
    print(f"Reduction: -{asr_drop:.1f}%")
else:
    print("Poisoning/Defense rows missing for TriviaQA")

print("\n=== DATA FOR TABLE 3: MIA DEFENSE (TriviaQA & PubMedQA) ===")
# Metric: Average "fill-in-the-blank" completion rate.
# The `mba_member` column contains the average percentage of correctly filled blanks for member docs.
# The `mba_non_member` column contains the rate for non-members (control group).
# We report `mba_member` as the "Inference Accuracy" or "Leakage Rate".

for ds in ['triviaqa', 'pubmedqa']:
    # MIA Only - Baseline
    m_base_rows = df[(df['dataset'] == ds) & (df['mba'] == True) & (df['dp'] == False)]
    # MIA + DP - Defense
    m_def_rows = df[(df['dataset'] == ds) & (df['mba'] == True) & (df['dp'] == True)]
    
    if not m_base_rows.empty and not m_def_rows.empty:
        m_base = m_base_rows.iloc[0]
        m_def = m_def_rows.iloc[0]
        
        # Leakage Rate (Member Accuracy)
        acc_base = m_base['mba_member']
        acc_def = m_def['mba_member']
        
        # Privacy Gain = Drop in Leakage Rate
        gain = acc_base - acc_def
        
        print(f"DATASET: {ds}")
        print(f"  No Defense Leakage (Member): {acc_base:.1f}%")
        print(f"  DP Defense Leakage (Member): {acc_def:.1f}%")
        print(f"  Privacy Gain (Leakage Reduction): {gain:.1f}%")
    else:
        print(f"{ds}: Missing MIA Rows")

print("\n=== DATA FOR TABLE 4: ADO RECOVERY (TriviaQA) ===")
# Comparison: Static Defense (TrustRAG/DP) vs ADO
# ADO row should have `ado`=True and potentially attacks enabled (depending on how it was logged)

# 1. Poisoning Comparison
tqa_static_poison = df[(df['dataset'] == 'triviaqa') & (df['poisoning'] == True) & (df['trustrag'] == True)]
tqa_ado = df[(df['dataset'] == 'triviaqa') & (df['ado'] == True)]

if not tqa_static_poison.empty and not tqa_ado.empty:
    st_p = tqa_static_poison.iloc[0]
    ado_p = tqa_ado.iloc[0] 
    
    print(f"POISONING SCENARIO:")
    print(f"  Static (TrustRAG) Recall: {st_p['context_recall']:.3f} | ASR: {st_p['poison_asr']}%")
    # Note: Using ADO context_recall. If it's 0.0 in the CSV, it will print 0.0.
    print(f"  ADO Recall: {ado_p['context_recall']:.3f} | ASR: {ado_p['poison_asr']}%") 
    
    if st_p['context_recall'] > 0:
        gain = ((ado_p['context_recall'] - st_p['context_recall']) / st_p['context_recall']) * 100
        print(f"  Utility Gain: {gain:+.1f}%")

# 2. MIA Comparison
tqa_static_mia = df[(df['dataset'] == 'triviaqa') & (df['mba'] == True) & (df['dp'] == True)]

if not tqa_static_mia.empty and not tqa_ado.empty:
    st_m = tqa_static_mia.iloc[0]
    ado_m = tqa_ado.iloc[0]
    
    print(f"MIA SCENARIO:")
    print(f"  Static (DP) Recall: {st_m['context_recall']:.3f} | Leakage: {st_m['mba_member']:.1f}%")
    print(f"  ADO Recall: {ado_m['context_recall']:.3f} | Leakage: {ado_m['mba_member']:.1f}%")
    
    if st_m['context_recall'] > 0:
        gain = ((ado_m['context_recall'] - st_m['context_recall']) / st_m['context_recall']) * 100
        print(f"  Utility Gain: {gain:+.1f}%")

=== DATA FOR TABLE 1: UTILITY COLLAPSE ===
DATASET: nq
  Baseline Recall: 0.593 | Relevancy: 0.930
  FullStack Recall: 0.350 | Relevancy: 0.863
  Recall Drop: -41.0%
------------------------------
DATASET: pubmedqa
  Baseline Recall: 0.625 | Relevancy: 0.833
  FullStack Recall: 0.335 | Relevancy: 0.583
  Recall Drop: -46.4%
------------------------------
DATASET: triviaqa
  Baseline Recall: 0.575 | Relevancy: 0.700
  FullStack Recall: 0.336 | Relevancy: 0.690
  Recall Drop: -41.6%
------------------------------

=== DATA FOR TABLE 2: POISONING DEFENSE (TriviaQA) ===
Baseline ASR: 56.0%
TrustRAG ASR: 0.0%
Reduction: -56.0%

=== DATA FOR TABLE 3: MIA DEFENSE (TriviaQA & PubMedQA) ===
DATASET: triviaqa
  No Defense Leakage (Member): 29.5%
  DP Defense Leakage (Member): 16.8%
  Privacy Gain (Leakage Reduction): 12.7%
DATASET: pubmedqa
  No Defense Leakage (Member): 37.0%
  DP Defense Leakage (Member): 19.3%
  Privacy Gain (Leakage Reduction): 17.7%

=== DATA FOR TABLE 4: ADO RECOVERY (Triv

In [17]:
import pandas as pd

# Load the summary CSV
df = pd.read_csv("defence_results_summary_20260201.csv")


print("=== DATA FOR TABLE 1: UTILITY COLLAPSE (MAIN AND APPENDIX CONTEXT) ===")
for ds in ['nq', 'pubmedqa', 'triviaqa']:
    # 1. Baseline: No defenses
    base_rows = df[
        (df['dataset'] == ds) &
        (df['dp'] == False) &
        (df['trustrag'] == False) &
        (df['av'] == False) &
        (df['poisoning'] == False) &
        (df['mba'] == False) &
        (df['ado'] == False)
    ]

    if base_rows.empty:
        print(f"{ds}: No Baseline Row Found")
        continue

    # If multiple baselines exist, pick the one with highest contextual recall
    base = base_rows.sort_values("context_recall", ascending=False).iloc[0]

    # 2. Full Stack: any row with at least one of DP / TrustRAG / AV on (no ADO)
    subset = df[
        (df['dataset'] == ds) &
        ((df['dp']) | (df['trustrag']) | (df['av'])) &
        (df['ado'] == False)
    ]

    if subset.empty:
        print(f"{ds}: No Defense Rows Found")
        continue

    # Heaviest stack proxy: lowest contextual recall
    full = subset.loc[subset['context_recall'].idxmin()]

    recall_drop = ((base['context_recall'] - full['context_recall']) /
                   base['context_recall']) * 100

    print(f"\nDATASET: {ds}")
    print(f"  Baseline - "
          f"Recall: {base['context_recall']:.3f}, "
          f"AnsRel: {base['answer_relevancy']:.3f}, "
          f"Faith: {base['faithfulness']:.3f}, "
          f"CtxRel: {base['context_relevancy']:.3f}")
    print(f"  FullStack - "
          f"Recall: {full['context_recall']:.3f}, "
          f"AnsRel: {full['answer_relevancy']:.3f}, "
          f"Faith: {full['faithfulness']:.3f}, "
          f"CtxRel: {full['context_relevancy']:.3f}")
    print(f"  Recall Drop: -{recall_drop:.1f}%")
    print("-" * 50)


print("\n=== DATA FOR TABLE 2: POISONING DEFENSE (TRIVIAQA) ===")
# Poisoning Only - Baseline (no TrustRAG)
p_base_rows = df[
    (df['dataset'] == 'triviaqa') &
    (df['poisoning'] == True) &
    (df['trustrag'] == False) &
    (df['ado'] == False)
]

# Poisoning + TrustRAG - Static Defense
p_def_rows = df[
    (df['dataset'] == 'triviaqa') &
    (df['poisoning'] == True) &
    (df['trustrag'] == True) &
    (df['ado'] == False)
]

if not p_base_rows.empty and not p_def_rows.empty:
    p_base = p_base_rows.iloc[0]
    p_def = p_def_rows.iloc[0]

    asr_drop = p_base['poison_asr'] - p_def['poison_asr']

    print("TRIVIAQA POISONING:")
    print(f"  No Defense - "
          f"ASR: {p_base['poison_asr']:.1f}%, "
          f"Recall: {p_base['context_recall']:.3f}, "
          f"AnsRel: {p_base['answer_relevancy']:.3f}")
    print(f"  TrustRAG - "
          f"ASR: {p_def['poison_asr']:.1f}%, "
          f"Recall: {p_def['context_recall']:.3f}, "
          f"AnsRel: {p_def['answer_relevancy']:.3f}")
    print(f"  ASR Reduction: -{asr_drop:.1f} percentage points")
else:
    print("Poisoning/Defense rows missing for TriviaQA")


print("\n=== DATA FOR TABLE 3: MIA DEFENSE (TRIVIAQA & PUBMEDQA) ===")
# mba_member = leakage rate (member accuracy)

for ds in ['triviaqa', 'pubmedqa']:
    # MIA Only - Baseline (no DP)
    m_base_rows = df[
        (df['dataset'] == ds) &
        (df['mba'] == True) &
        (df['dp'] == False) &
        (df['ado'] == False)
    ]
    # MIA + DP - Static Defense
    m_def_rows = df[
        (df['dataset'] == ds) &
        (df['mba'] == True) &
        (df['dp'] == True) &
        (df['ado'] == False)
    ]

    if not m_base_rows.empty and not m_def_rows.empty:
        m_base = m_base_rows.iloc[0]
        m_def = m_def_rows.iloc[0]

        acc_base = m_base['mba_member']
        acc_def = m_def['mba_member']
        gain = acc_base - acc_def

        print(f"\nDATASET: {ds}")
        print(f"  No Defense - Leakage (member): {acc_base:.1f}% "
              f"| Recall: {m_base['context_recall']:.3f} "
              f"| AnsRel: {m_base['answer_relevancy']:.3f}")
        print(f"  DP-RAG - Leakage (member): {acc_def:.1f}% "
              f"| Recall: {m_def['context_recall']:.3f} "
              f"| AnsRel: {m_def['answer_relevancy']:.3f}")
        print(f"  Privacy Gain (Leakage Reduction): {gain:.1f} percentage points")
    else:
        print(f"{ds}: Missing MIA rows")


print("\n=== DATA FOR TABLE 4: ADO RECOVERY (TRIVIAQA) ===")
# ADO rows: ado == True (you may have multiple; pick ones with poisoning/MIA)

# 1. Poisoning scenario: Static TrustRAG vs ADO under poisoning
tqa_static_poison = df[
    (df['dataset'] == 'triviaqa') &
    (df['poisoning'] == True) &
    (df['trustrag'] == True) &
    (df['ado'] == False)
]

tqa_ado_poison = df[
    (df['dataset'] == 'triviaqa') &
    (df['poisoning'] == True) &
    (df['ado'] == True)
]

if not tqa_static_poison.empty and not tqa_ado_poison.empty:
    st_p = tqa_static_poison.iloc[0]
    ado_p = tqa_ado_poison.iloc[0]

    print("POISONING SCENARIO (TRIVIAQA):")
    print(f"  Static (TrustRAG) - Recall: {st_p['context_recall']:.3f}, "
          f"AnsRel: {st_p['answer_relevancy']:.3f}, "
          f"ASR: {st_p['poison_asr']:.1f}%")
    print(f"  ADO - Recall: {ado_p['context_recall']:.3f}, "
          f"AnsRel: {ado_p['answer_relevancy']:.3f}, "
          f"ASR: {ado_p['poison_asr']:.1f}%")

    if st_p['context_recall'] > 0:
        gain = ((ado_p['context_recall'] - st_p['context_recall']) /
                st_p['context_recall']) * 100
        print(f"  Utility Gain (Recall): {gain:+.1f}%")
else:
    print("Missing ADO or static poisoning rows for TriviaQA")


# 2. MIA scenario: Static DP vs ADO under MIA
tqa_static_mia = df[
    (df['dataset'] == 'triviaqa') &
    (df['mba'] == True) &
    (df['dp'] == True) &
    (df['ado'] == False)
]

tqa_ado_mia = df[
    (df['dataset'] == 'triviaqa') &
    (df['mba'] == True) &
    (df['ado'] == True)
]

if not tqa_static_mia.empty and not tqa_ado_mia.empty:
    st_m = tqa_static_mia.iloc[0]
    ado_m = tqa_ado_mia.iloc[0]

    print("\nMIA SCENARIO (TRIVIAQA):")
    print(f"  Static (DP-RAG) - Recall: {st_m['context_recall']:.3f}, "
          f"AnsRel: {st_m['answer_relevancy']:.3f}, "
          f"Leakage: {st_m['mba_member']:.1f}%")
    print(f"  ADO - Recall: {ado_m['context_recall']:.3f}, "
          f"AnsRel: {ado_m['answer_relevancy']:.3f}, "
          f"Leakage: {ado_m['mba_member']:.1f}%")

    if st_m['context_recall'] > 0:
        gain = ((ado_m['context_recall'] - st_m['context_recall']) /
                st_m['context_recall']) * 100
        print(f"  Utility Gain (Recall): {gain:+.1f}%")
else:
    print("Missing ADO or static MIA rows for TriviaQA")


=== DATA FOR TABLE 1: UTILITY COLLAPSE (MAIN AND APPENDIX CONTEXT) ===

DATASET: nq
  Baseline - Recall: 0.593, AnsRel: 0.930, Faith: 0.740, CtxRel: 0.489
  FullStack - Recall: 0.350, AnsRel: 0.863, Faith: 0.720, CtxRel: 0.305
  Recall Drop: -41.0%
--------------------------------------------------

DATASET: pubmedqa
  Baseline - Recall: 0.625, AnsRel: 0.833, Faith: 0.643, CtxRel: 0.424
  FullStack - Recall: 0.335, AnsRel: 0.583, Faith: 0.720, CtxRel: 0.205
  Recall Drop: -46.4%
--------------------------------------------------

DATASET: triviaqa
  Baseline - Recall: 0.575, AnsRel: 0.700, Faith: 0.790, CtxRel: 0.514
  FullStack - Recall: 0.336, AnsRel: 0.690, Faith: 0.770, CtxRel: 0.232
  Recall Drop: -41.6%
--------------------------------------------------

=== DATA FOR TABLE 2: POISONING DEFENSE (TRIVIAQA) ===
TRIVIAQA POISONING:
  No Defense - ASR: 56.0%, Recall: 0.585, AnsRel: 0.687
  TrustRAG - ASR: 0.0%, Recall: 0.574, AnsRel: 0.710
  ASR Reduction: -56.0 percentage points

===

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37 entries, 0 to 36
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   dp                 37 non-null     bool   
 1   trustrag           37 non-null     bool   
 2   av                 37 non-null     bool   
 3   poisoning          37 non-null     bool   
 4   mba                37 non-null     bool   
 5   ado                37 non-null     bool   
 6   answer_relevancy   37 non-null     float64
 7   faithfulness       37 non-null     float64
 8   context_relevancy  37 non-null     float64
 9   context_recall     37 non-null     float64
 10  poison_acr         37 non-null     int64  
 11  mba_member         37 non-null     float64
 12  mba_non_member     37 non-null     float64
 13  poison_success     37 non-null     int64  
 14  poison_total       37 non-null     int64  
 15  dataset            37 non-null     object 
 16  poison_asr         37 non-nu

In [31]:
df[(df['dataset'] == 'nq') & (df['poisoning'] == True)]

,dp,trustrag,av,poisoning,mba,ado,answer_relevancy,faithfulness,context_relevancy,context_recall,poison_acr,mba_member,mba_non_member,poison_success,poison_total,dataset,poison_asr
1,False,False,False,True,False,False,0.91,0.73,0.506,0.598,0,0.0,0.0,35,50,nq,70.0
2,False,True,False,True,False,False,0.89,0.79,0.462,0.614,0,0.0,0.0,0,50,nq,0.0
3,False,False,True,True,False,False,0.74,0.70,0.439,0.583,0,0.0,0.0,36,50,nq,72.0


In [23]:
import pandas as pd

# Your existing df is already loaded with 37 rows
# Filter for static defense experiments only (no attacks active)
static_df = df[(df['poisoning'] == False) & 
               (df['mba'] == False) & 
               (df['ado'] == False)].copy()

# Create configuration labels
def get_config_label(row):
    d, t, a = row['dp'], row['trustrag'], row['av']
    if not d and not t and not a:
        return "Baseline (none)"
    elif d and not t and not a:
        return "DP-RAG only (D)"
    elif not d and t and not a:
        return "TrustRAG only (T)"
    elif not d and not t and a:
        return "AV-Filtering only (A)"
    elif d and t and not a:
        return "D + T"
    elif d and not t and a:
        return "D + A"
    elif not d and t and a:
        return "T + A"
    elif d and t and a:
        return "Full Stack (D+T+A)"
    return "Unknown"

static_df['Configuration'] = static_df.apply(get_config_label, axis=1)

# Map dataset names if needed (assuming they're lowercase in your df)
dataset_map = {
    'nq': 'Natural Questions',
    'pubmedqa': 'PubMedQA', 
    'triviaqa': 'TriviaQA'
}
static_df['Dataset'] = static_df['dataset'].map(dataset_map)

# Pivot to get configurations as rows, datasets as columns with metrics
metrics = ['context_recall', 'context_relevancy', 'answer_relevancy', 'faithfulness']
metric_short = {'context_recall': 'CR', 'context_relevancy': 'CP', 
                'answer_relevancy': 'AR', 'faithfulness': 'F'}

# Order configurations logically
config_order = [
    "Baseline (none)",
    "DP-RAG only (D)",
    "TrustRAG only (T)",
    "AV-Filtering only (A)",
    "D + T",
    "D + A",
    "T + A",
    "Full Stack (D+T+A)"
]

# Generate LaTeX table
latex_lines = []
latex_lines.append(r"\begin{table*}[t]")
latex_lines.append(r"\centering")
latex_lines.append(r"\small")
latex_lines.append(r"\setlength{\tabcolsep}{3pt}")
latex_lines.append(r"\caption{Complete utility evaluation across defense permutations. CR = Contextual Recall, CP = Contextual Precision, AR = Answer Relevancy, F = Faithfulness.}")
latex_lines.append(r"\label{tab:defense-permutations}")
latex_lines.append(r"\begin{tabular}{@{}l " + "c"*12 + r"@{}}")
latex_lines.append(r"\toprule")
latex_lines.append(r" & \multicolumn{4}{c}{\textbf{Natural Questions}} & \multicolumn{4}{c}{\textbf{PubMedQA}} & \multicolumn{4}{c}{\textbf{TriviaQA}} \\")
latex_lines.append(r"\cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13}")
latex_lines.append(r" & CR & CP & AR & F & CR & CP & AR & F & CR & CP & AR & F \\")
latex_lines.append(r"\midrule")

# For each configuration, extract values across datasets
for config in config_order:
    conf_data = static_df[static_df['Configuration'] == config]
    
    # Skip if config not found
    if len(conf_data) == 0:
        continue
        
    # Get values for each dataset
    datasets = ['Natural Questions', 'PubMedQA', 'TriviaQA']
    values = []
    
    for ds in datasets:
        ds_data = conf_data[conf_data['Dataset'] == ds]
        if len(ds_data) == 0:
            # If missing, use placeholder or compute from available
            row_vals = ['0.000'] * 4
        else:
            row = ds_data.iloc[0]
            row_vals = [f"{row[m]:.3f}" for m in metrics]
        values.extend(row_vals)
    
    # Bold the full stack row
    if "Full Stack" in config:
        config_disp = r"\textbf{" + config + "}"
        values = [r"\textbf{" + v + "}" for v in values]
    else:
        config_disp = config
    
    latex_lines.append(f"{config_disp} & {' & '.join(values)} \\\\")

latex_lines.append(r"\bottomrule")
latex_lines.append(r"\end{tabular}")
latex_lines.append(r"\end{table*}")

# Print the table
print("\n".join(latex_lines))

\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{3pt}
\caption{Complete utility evaluation across defense permutations. CR = Contextual Recall, CP = Contextual Precision, AR = Answer Relevancy, F = Faithfulness.}
\label{tab:defense-permutations}
\begin{tabular}{@{}l cccccccccccc@{}}
\toprule
 & \multicolumn{4}{c}{\textbf{Natural Questions}} & \multicolumn{4}{c}{\textbf{PubMedQA}} & \multicolumn{4}{c}{\textbf{TriviaQA}} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13}
 & CR & CP & AR & F & CR & CP & AR & F & CR & CP & AR & F \\
\midrule
Baseline (none) & 0.593 & 0.489 & 0.930 & 0.740 & 0.625 & 0.424 & 0.833 & 0.643 & 0.575 & 0.514 & 0.700 & 0.790 \\
DP-RAG only (D) & 0.361 & 0.321 & 0.863 & 0.770 & 0.398 & 0.270 & 0.793 & 0.680 & 0.359 & 0.264 & 0.750 & 0.820 \\
TrustRAG only (T) & 0.586 & 0.470 & 0.860 & 0.720 & 0.584 & 0.418 & 0.803 & 0.712 & 0.567 & 0.509 & 0.720 & 0.810 \\
AV-Filtering only (A) & 0.592 & 0.449 & 0.780 & 0.770 & 0.591 & 0.390 & 0.760 & 0.666 & 